In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

BASE_PATH = "../"

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"


In [2]:
import subprocess
import pandas as pd
from transformers import pipeline
import json
import time
import glob
from tqdm import tqdm

In [3]:
chunk_duration = 300 # 300 secs = 5 mins btw

In [4]:
def split_audio(input_path, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        
    for f in glob.glob(os.path.join(output_directory, "*.mp3")):
        os.remove(f)

    cmd = [
        "ffmpeg", "-i", input_path, 
        "-f", "segment",
        "-segment_time", str(chunk_duration),
        "-c", "copy", os.path.join(output_directory, "%03d.mp3")
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    return sorted(glob.glob(os.path.join(output_directory, "*.mp3")))



In [8]:
from faster_whisper import WhisperModel

def transcribe_lecture(audio_path, save_path):
    model = WhisperModel("medium", device="cuda", compute_type="float16", use_auth_token=HF_TOKEN)
    
    segments, info = model.transcribe(audio_path, beam_size=5, language="ru", vad_filter=True)
    
    results = []
    for segment in segments:
        results.append({
            "start": segment.start,
            "end": segment.end,
            "text": segment.text
        })
    
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
        
    return pd.DataFrame(results)

In [9]:
import os
import glob
from tqdm import tqdm

RAW_DIR = BASE_PATH + "data/raw"
TRANSCRIPTIONS_BASE = BASE_PATH + "data/transcriptions"
subdirs = sorted([d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))])
print(f"Found {len(subdirs)} directories: {subdirs}")


Found 7 directories: ['00', '01', '02', '03', '04', '05', '06']


In [10]:
subdir = subdirs[0] + "_"
input_path_dir = os.path.join(RAW_DIR, subdirs[0])
output_path_dir = os.path.join(TRANSCRIPTIONS_BASE, subdir)
output_chunks_dir = os.path.join(output_path_dir, "chunks")
save_json_path = os.path.join(output_path_dir, "lecture_raw.json")

os.makedirs(output_chunks_dir, exist_ok=True)

mp3_files = glob.glob(os.path.join(input_path_dir, "*.mp3"))

audio_file = mp3_files[0]

df_transcript = transcribe_lecture(
    audio_path=audio_file,
    save_path=save_json_path,
)

df_transcript.head()

KeyboardInterrupt: 

In [ ]:
for subdir in tqdm(subdirs, desc="Transcription"):
    input_path_dir = os.path.join(RAW_DIR, subdir)
    output_path_dir = os.path.join(TRANSCRIPTIONS_BASE, subdir)
    output_chunks_dir = os.path.join(output_path_dir, "chunks")
    save_json_path = os.path.join(output_path_dir, "lecture_raw.json")
    
    os.makedirs(output_chunks_dir, exist_ok=True)
    
    mp3_files = glob.glob(os.path.join(input_path_dir, "*.mp3"))
    if not mp3_files:
        print(f"Warning: no .mp3 found in {subdir}")
        continue
    if len(mp3_files) > 1:
        print(f"Warning: found multiple .mp3 files in {subdir}")
    
    audio_file = mp3_files[0]
    
    try:
        transcribe_lecture(
            audio_path=audio_file,
            save_path=save_json_path,
        )
    except Exception as e:
        print(f"Error during processing {subdir}: {e}")


Transcription: 100%|██████████| 7/7 [1:21:48<00:00, 701.26s/it]

Transcription completed!


: 